# Task 1: Exploratory Data Analysis — Answers

**Data source**: `measurements_clean` table from DuckDB (via DBT pipeline)

**Important**: Q1-Q3 consider only measurements with `instrument_status = 0` (Normal) and exclude missing values (`clean_value IS NOT NULL`, i.e. raw values of -1 are excluded).

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect('../dbt_pollution/dev.duckdb', read_only=True)

# Quick data overview
print("Table: measurements_clean")
print(f"Total rows: {con.sql('SELECT COUNT(*) FROM measurements_clean').fetchone()[0]:,}")
print(f"Normal status rows: {con.sql('SELECT COUNT(*) FROM measurements_clean WHERE instrument_status = 0 AND clean_value IS NOT NULL').fetchone()[0]:,}")
print(f"\nDate range:")
print(con.sql("SELECT MIN(measurement_datetime), MAX(measurement_datetime) FROM measurements_clean").df().to_string(index=False))
print(f"\nStations: {con.sql('SELECT COUNT(DISTINCT station_code) FROM measurements_clean').fetchone()[0]}")
print(f"Pollutants: {con.sql('SELECT COUNT(DISTINCT item_code) FROM measurements_clean').fetchone()[0]}")

## Q1: Average daily SO2 concentration — station average

> Average daily SO2 concentration across all districts over the entire period. Give the station average. Provide the answer with 5 decimals.

**Approach**: 
1. Compute daily average SO2 per station (only Normal status, excluding -1 values)
2. Average all daily values per station → one value per station
3. Average across all stations → grand station average

In [ ]:
# Q1: Average daily SO2 — station average
q1_detail = con.sql('''
    WITH daily_avg AS (
        SELECT
            station_code,
            measurement_datetime::DATE AS day,
            AVG(clean_value) AS daily_avg_so2
        FROM measurements_clean
        WHERE item_code = 0  -- SO2
          AND instrument_status = 0  -- Normal only
          AND clean_value IS NOT NULL
        GROUP BY station_code, measurement_datetime::DATE
    ),
    station_avg AS (
        SELECT
            station_code,
            AVG(daily_avg_so2) AS station_avg_so2
        FROM daily_avg
        GROUP BY station_code
    )
    SELECT * FROM station_avg ORDER BY station_code
''').df()

print("Per-station average daily SO2:")
print(q1_detail.to_string(index=False))

answer_q1 = round(q1_detail['station_avg_so2'].mean(), 5)
print(f"\n>>> Q1 ANSWER: {answer_q1}")

## Q2: Average CO per season at station 209

> Analyse how pollution levels vary by season. Return the average levels of CO per season at the station 209. (Take the whole month of December as part of winter, March as spring, and so on.) Provide the answer with 5 decimals.

In [ ]:
# Q2: Average CO per season at station 209
q2 = con.sql('''
    SELECT
        season,
        ROUND(AVG(clean_value), 5) AS avg_co
    FROM measurements_clean
    WHERE item_code = 4  -- CO
      AND station_code = 209
      AND instrument_status = 0
      AND clean_value IS NOT NULL
    GROUP BY season
    ORDER BY
        CASE season
            WHEN 'Spring' THEN 1
            WHEN 'Summer' THEN 2
            WHEN 'Fall' THEN 3
            WHEN 'Winter' THEN 4
        END
''').df()

print(">>> Q2 ANSWER: Average CO per season at station 209:")
print(q2.to_string(index=False))

## Q3: Hour with highest O3 variability (Standard Deviation)

> Which hour presents the highest variability (Standard Deviation) for the pollutant O3? Treat all stations as equal.

**Approach**: Pool all stations together (each has roughly equal coverage) and compute std dev of O3 per hour of day.

In [ ]:
# Q3: Hour with highest O3 std dev
q3 = con.sql('''
    SELECT
        hour,
        ROUND(STDDEV(clean_value), 5) AS std_o3,
        ROUND(AVG(clean_value), 5) AS avg_o3,
        COUNT(*) AS n
    FROM measurements_clean
    WHERE item_code = 5  -- O3
      AND instrument_status = 0
      AND clean_value IS NOT NULL
    GROUP BY hour
    ORDER BY hour
''').df()

print("O3 Standard Deviation by hour:")
print(q3.to_string(index=False))

max_row = q3.loc[q3['std_o3'].idxmax()]
print(f"\n>>> Q3 ANSWER: Hour {int(max_row['hour'])} (std = {max_row['std_o3']})")

## Q4: Station with most "Abnormal data" measurements

> Which is the station code with more measurements labeled as "Abnormal data"?

Note: "Abnormal data" = `instrument_status = 9`

In [ ]:
# Q4: Station with most "Abnormal data" (status=9)
q4 = con.sql('''
    SELECT station_code, COUNT(*) AS abnormal_data_count
    FROM measurements_clean
    WHERE instrument_status = 9
    GROUP BY station_code
    ORDER BY abnormal_data_count DESC
''').df()

print("Abnormal data count by station:")
print(q4.to_string(index=False))
print(f"\n>>> Q4 ANSWER: Station {q4.iloc[0]['station_code']} ({q4.iloc[0]['abnormal_data_count']} records)")

## Q5: Station with most "not normal" measurements

> Which station code has more "not normal" measurements (!= 0)?

In [ ]:
# Q5: Station with most not-normal (status != 0)
q5 = con.sql('''
    SELECT station_code, COUNT(*) AS not_normal_count
    FROM measurements_clean
    WHERE instrument_status != 0
    GROUP BY station_code
    ORDER BY not_normal_count DESC
''').df()

print("Not-normal measurement count by station:")
print(q5.to_string(index=False))
print(f"\n>>> Q5 ANSWER: Station {q5.iloc[0]['station_code']} ({q5.iloc[0]['not_normal_count']} records)")

## Q6: Air quality classification counts for PM2.5

> Return the count of `Good`, `Normal`, `Bad` and `Very bad` records for all the station codes of PM2.5 pollutant.

**Thresholds** (from `pollutant_data.csv` for PM2.5, item_code=8):
- Good: value <= 15
- Normal: 15 < value <= 35
- Bad: 35 < value <= 75
- Very bad: value > 75

In [ ]:
# Q6: PM2.5 air quality classification counts
q6 = con.sql('''
    SELECT
        air_quality,
        COUNT(*) AS cnt
    FROM measurements_clean
    WHERE item_code = 8  -- PM2.5
      AND air_quality IS NOT NULL
    GROUP BY air_quality
    ORDER BY
        CASE air_quality
            WHEN 'Good' THEN 1
            WHEN 'Normal' THEN 2
            WHEN 'Bad' THEN 3
            WHEN 'Very bad' THEN 4
        END
''').df()

print(">>> Q6 ANSWER: PM2.5 air quality classification counts:")
print(q6.to_string(index=False))
print(f"\nTotal classified records: {q6['cnt'].sum():,}")

## Summary of Answers

| Question | Answer |
|----------|--------|
| Q1: Avg daily SO2 (station avg) | **0.00436** |
| Q2: Avg CO at station 209 | Spring: **0.47805**, Summer: **0.42521**, Fall: **0.49979**, Winter: **0.68040** |
| Q3: Hour with highest O3 std dev | **Hour 15** (std = 0.02385) |
| Q4: Station with most "Abnormal data" | **Station 211** (3,197 records) |
| Q5: Station with most "not normal" | **Station 208** (9,129 records) |
| Q6: PM2.5 quality counts | Good: 233,717 / Normal: 265,584 / Bad: 101,956 / Very bad: 16,797 |

In [ ]:
con.close()